# Kết quả cuối E0–E4 — Phan Trọng Hoàn

E1/E2/E3 và E4 đủ300 bước. Notebook này đọc evidence thật đã lưu, không tải model hoặc train; chạy được CPU tại repo. Không coi các cell chưa thực thi trong notebook này là lượt suy luận mới. Workflow huấn luyện: lab_day10_continue300.ipynb. Nguồn Colab do sinh viên cung cấp: extended300_results (1).zip, extended300_figures.zip và notebook tải về (1)/(2), giữ riêng local. Xem REPORT.md để biết resume, mất backup E4 v1 và chạy lại v2, AMP scale khác và model_revision=null.


In [ ]:
from pathlib import Path
import json
ROOT = Path.cwd()
assert (ROOT / 'report/evidence/e3.json').exists(), 'Chạy từ thư mục gốc repo.'
names = ['e0', 'e1', 'e2', 'e3', 'e4_data_size_0.1', 'e4_data_size_1.0']
evidence = {name: json.loads((ROOT / 'report/evidence' / f'{name}.json').read_text(encoding='utf-8')) for name in names}
for name, d in evidence.items():
    assert d['n_images'] == 311 and d['missing_predictions'] == 0
    print(name, 'F1=', d['micro']['f1'], 'CI95=', d['f1_ci95'], 'valid=', d['parse_valid_rate'])


## Chọn đối chứng bằng val
E3 được chọn trước khi chấm test E4. E4 chỉ đổi fraction10% so100%, cùng300 bước/horizon300; nhánh10% khởi tạo từ model gốc.


In [ ]:
from src.results import choose_by_val
base = choose_by_val(ROOT / 'report/validation')
assert base == 'e3'
for name in ('e3', 'e4_data_size_0.1', 'e4_data_size_1.0'):
    assert evidence[name]['meta']['actual_steps'] == 300
delta = evidence['e4_data_size_1.0']['micro']['f1'] - evidence['e4_data_size_0.1']['micro']['f1']
print('F1 giảm khi dùng10%:', delta)
print(json.loads((ROOT / 'report/extended/analysis/architecture.json').read_text()))


## Sáu ảnh từ prediction E3 300 bước và bẫy tọa độ
Manifest do scorer gốc ghép box, không suy luận lại. GT xanh, prediction đỏ. Các ảnh được chọn là ví dụ, không đại diện tỷ lệ lỗi toàn tập.


In [ ]:
try:
    from IPython.display import display, Image
except ImportError:
    # Terminal CPU check: verify files without installing notebook dependencies.
    from PIL import Image as PILImage
    def Image(filename):
        with PILImage.open(filename) as image:
            image.verify()
        return filename
    def display(value):
        print('Image verified:', value)
manifest = json.loads((ROOT / 'report/analysis/figure_manifest.json').read_text(encoding='utf-8'))
for example in manifest['examples']:
    print(example['image_id'], example['caption'])
    display(Image(filename=str(ROOT / example['figure'])))
coord = json.loads((ROOT / 'report/analysis/coord_scale_check.json').read_text(encoding='utf-8'))
assert abs(coord['correct_norm1000_f1'] - evidence['e3']['micro']['f1']) < 1e-12
print(coord)


## Kiểm tra và giới hạn
Test gốc:67 đạt,1 bỏ qua,1 lỗi do marker REPORT trái checker. Checker0 FAIL0 WARN tại lần kiểm tra ghi trong report/analysis/verification.json. Không sửa scripts/tests gốc. Một seed, CI bootstrap theo ảnh, AMP scale khác nhau và thiếu model revision hạn chế diễn giải. E5 chưa làm. Báo cáo200/evidence lịch sử ở report/history/step0200; biểu đồ loss cũ được ghi rõ nguồn, không thay cho log run300.
